# Expert-CFG + RAG 평가 — VQA-RAD · SLAKE (Phi-3.5V-Med)

논문 *Uncertainty-Driven Expert Control* (arXiv:2507.09209)의 RAG · Expert-CFG 를 **같은 파이프라인으로 VQA-RAD 와 SLAKE 에 적용**해 베이스라인 · w/ RAG · 5% Expert-RAG · 5% Expert-CFG 를 한 표로 비교한다. 위쪽에 논문 결과를 정리했다.

**실행 방법**: 설정 셀의 `DATASET` 을 `'vqarad'` 로 한 번, `'slake'` 로 한 번 처음부터 실행한다(어댑터가 달라 모델을 다시 로드해야 한다).
**데이터셋을 바꿀 때는 런타임을 재시작**하고 모든 셀을 한 번씩만 실행한다. 같은 세션에서 `DATASET` 만 바꿔 다시 실행하면 모델이 GPU에 두 벌 올라가 문항당 속도가 수십 배 느려진다.
실험 결과는 문항 단위로 Drive에 저장되므로 끊겨도 같은 설정으로 다시 실행하면 이어서 계산한다(엔트로피 셀도 100문항마다 저장).

**구성**: 3. 실험(GPU) → 4. 결과표 → 4-2. **RAG 진단 · 검색 방식 비교**(GPU 불필요, 저장된 결과만 읽음) → 5. 종합 비교(두 데이터셋을 다 돌린 뒤).
검색 방식 변형(A union+QA / B 이미지→텍스트+QA / C PEIR+ROCO)은 설정 셀의 `KB`·`KB_TAG` 로 고르고, 변형별 검색 파일은 4-2 절의 빌더 셀로 만든다.

**규칙**: 어댑터는 train 으로만 학습하고 val 로 고른 체크포인트다(VQA-RAD `checkpoint-257`, SLAKE `checkpoint-380`). test 는 최종 평가에서만 읽는다. RAG · Expert 행에서 test 로 고른 설정은 없다. 정답을 쓰는 행은 **규칙 완화(oracle)** 로 따로 표기한다.

## 논문 결과 (Table 1, Phi-3.5V-Med 4.2B · Open / Closed / Overall)

| 방법 | VQA-RAD | SLAKE |
|---|---|---|
| Phi3.5V-Med (베이스라인) | 69.27 / 90.44 / **82.04** | 82.95 / 88.70 / **85.20** |
| w/ RAG (전 문항, top-1) | 70.39 / 88.24 / **81.15** | 82.79 / 89.90 / **85.58** |
| w/ 5% Expert-RAG | 72.06 / 90.44 / **83.37** | 83.56 / 89.90 / **86.05** |
| w/ 5% Expert-CFG | 74.86 / 91.18 / **84.70** | 85.27 / 91.59 / **87.75** |

참고(Phi3V-Med): 베이스라인 VQA-RAD 79.38 / SLAKE 83.98, w/ RAG 80.71 / 85.11, 5% Expert-RAG 80.27 / 85.29, 5% Expert-CFG 81.82 / 86.33 (overall). 다른 모델 참고: LLaVA-Med 13B VQA-RAD 72.66 / SLAKE 85.21, BioMed-VITAL 13B 78.85 / 91.30 (overall).

**RAG 검색 정확도 (Table 2, top-1 hit rate, VQA-RAD / SLAKE)**: Image 53.07 / 47.75, Text 53.07 / 54.57, Sum 53.07 / 54.26, Union 53.07 / 55.97.

**하이퍼파라미터**: 가이던스 γ=1.5(본문)이나 Table 7 에서는 γ=1.3 이 세 데이터셋 모두 최고, α(perturb 가중치)=0.01, β(attention 가중치)=3, 엔트로피 상위 **5%** 문항만 전문가 처리, CLIP score 0.6 미만은 대체로 무관한 참조.

> 위 수치는 arXiv HTML 본문에서 도구로 추출한 값이다. 베이스라인 82.04 / 85.20 은 이 프로젝트의 재현 목표와 일치해 신뢰하지만, 나머지 행은 **원문과 한 번 대조**할 것. Table 7 의 α·β·γ 스윕 수치는 의미가 불분명해 옮기지 않았다.

### 논문과 이 노트북의 차이 (결과 해석 시 반드시 고려)

| 항목 | 논문 | 이 노트북 |
|---|---|---|
| 검색 지식베이스 | ROCO 70,306 + PEIR 33,572 + 세 데이터셋 train QA 를 캡션으로 바꾼 것 (합계 10만 쌍 이상) | **PEIR 33,572 만** |
| 검색 방식 | BioMedCLIP 으로 지식베이스와 쿼리 인코딩, FAISS | BioMedCLIP 이미지 쿼리 × PEIR 캡션 텍스트 코사인 (`tools/build_rag_testjson.py`, 자체 검색) |
| w/ RAG | top-1 캡션을 전 문항에 | 같음 (top-1) |
| 하이라이트 | GPT-4 가 검색 텍스트에서 키워드를 뽑고 전문가가 수정, **open 문항은 정답을 그대로 전문가 어노테이션으로 사용** | 기본: PEIR `match_keyword` (정답 미사용). **oracle 행**: open 문항 하이라이트 = 정답 |
| 불확실성 | 예측 엔트로피 상위 5% | 같음 (베이스라인 생성 시 평균 토큰 엔트로피) |

**KB 선택**: 설정 셀의 `KB` 가 `'peir'`(기본) 면 PEIR 만, `'peir,roco,trainqa'` 면 ROCO(공개본 train 65,423 캡션)와 VQA-RAD · SLAKE train QA(텍스트로 단순 연결)를 더한다. 논문 KB 는 ROCO 70,306 + PEIR 33,572 + 세 데이터셋 train QA 이며, 우리는 PathVQA 가 없고 train QA → 캡션 변환이 논문과 다르므로 "자체 검색 + KB 구성"으로 표기한다. train QA 를 넣으면 같은 이미지의 train QA 가 검색되는 누수 가능성이 있어(VQA-RAD 는 train / test 이미지가 많이 겹침) 데이터 셀이 그 문항 수를 출력한다. KB 별 검색 파일은 `tools/build_rag_testjson.py --kb ...` 로 먼저 만든다.

## 실험 행과 규칙 표기

**기본 비교 행은 "논문 방식" 정답 주입 행이다**(`ExpertRAG-5-inject-v2`, `ExpertCFG-5-inject-v2`). 논문 5.2절은 open 문항에서 정답을 전문가 어노테이션으로 쓴다고 설명하고, 이 행이 그 원리를 따른다. 단 메인 Table 1이 이 방식인지는 원문에 명시되어 있지 않고(부록 C는 GPT-4 키워드 추출 + 전문가 검토), **test 정답이 입력에 들어가므로 `규칙 완화(oracle)`로 표기하며 규칙 준수 결과로 보고하지 않는다.** 규칙 준수 행(정답 미사용)은 비교 대조로 함께 보여 준다.

| 행 | 내용 | 규칙 |
|---|---|---|
| 베이스라인 | 어댑터만, 캡션 없음 (저장된 test 결과 파일) | 규칙 준수 |
| w/ RAG | 전 문항에 top-1 캡션, 하이라이트·CFG 없음 | 자체 검색 |
| w/ 5% RAG | 엔트로피 상위 5%만 top-1 캡션, 나머지는 베이스라인 | 자체 검색 |
| w/ 5% Expert-CFG | 상위 5%: 캡션 + 하이라이트 + CFG. **논문 식 7·9·10 (팀원 phi3_urp 와 같은 정의)**: 무조건부 가지에서 하이라이트 임베딩 ×α, 정상 가지 attention +log β, 무조건부 가지 attention −δ | **하이라이트: open = 검색 파일의 규칙 기반 하이라이트, closed = 질문과 내용어가 가장 많이 겹치는 캡션 문장(최대 2개)**. 정답 미사용 |
| w/ 5% Expert-RAG (정답 주입) | 상위 5%: open 문항 참조에 정답을 덧붙임(검색 캡션에 정답이 없을 때), 가이던스 없음 | **규칙 완화 (논문 방식 모사)** |
| w/ 5% Expert-CFG (정답 주입) | 위와 같은 참조에서 open 정답을 하이라이트 + 논문식 CFG. **closed 는 정답을 넣지 않고 하이라이트도 검색 파일의 n-gram 규칙을 쓴다(규칙 준수 행의 문장 규칙과 다르다)** | **규칙 완화 (논문 방식 모사)** |

정답을 쓰지 않는 규칙 준수 행에서 "5% Expert-RAG"는 5% RAG 와 같다(둘 다 같은 top-1 캡션을 넣을 뿐이고 가이던스가 없다). 그래서 별도 행을 두지 않았다. 논문의 Expert-RAG 와 RAG 의 차이는 전문가 주석(open 은 정답)이다.

나머지 95% 문항은 베이스라인 예측을 그대로 쓴다. **공식 채점은 원본(Phi 3.5V 방식: closed 정답 단어 포함, open 단어 recall)** 이다. S1+S2 (표기 차이만 허용하는 open 채점: 대소문자·하이픈·복수형·약어, 일반어 plane/weighted/mri 생략)는 Phi 의 VQA-RAD test 오답을 보고 만든 규칙이라 교수님 규칙 기준으로 규칙 완화에 해당하므로, "참고(규칙 완화)" 열로만 병기하고 공식 결과로 보고하지 않는다.

**정답 주입 형식** (`inject` 행, 5% 게이팅으로 선택된 문항에만 적용)

| 문항 | 프롬프트의 캡션 | 하이라이트 |
|---|---|---|
| open | top-1 캡션에 정답(대소문자 무시 부분 문자열)이 **없으면** 뒤에 `정답.`을 덧붙임. 있으면 그대로 | 정답 문자열 |
| closed | top-1 캡션 그대로 (정답 없음) | 검색 파일의 `highlights` (캡션과 질문이 공유하는 1~3-gram) |

입력은 `{캡션} <|image_1|>
{질문}` 이다. 하이라이트 문자열은 프롬프트에서 **처음 나오는 위치**로 찾으므로, 정답이 이미 캡션에 있으면 캡션 속 다른 QA 문장의 같은 단어에 마스크가 걸릴 수 있다(검증하지 않았다). 논문은 전문가가 참조 텍스트를 수정한다고 설명하고, 정답을 덧붙이는 이 형식이 그것과 같은지는 확인하지 못했다.


## 0. 환경

In [ ]:
# 환경 1) 저장소 클론 · Drive 마운트
REPO_BRANCH = 'handoff/phi35v-vqarad-slake-rag-cfg'      # ← 쓰려는 브랜치 이름 (vqa_scoring.py 와 processing_phi3_v_cfg.py 가 있는 브랜치)
import os, subprocess
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
if not os.path.isdir('/content/Expert-CFG'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, 'https://github.com/MinseongKim-dev/Expert-CFG.git', '/content/Expert-CFG'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Expert-CFG', 'fetch', 'origin'])
    subprocess.run(['git', '-C', '/content/Expert-CFG', 'checkout', REPO_BRANCH])
    subprocess.run(['git', '-C', '/content/Expert-CFG', 'pull', 'origin', REPO_BRANCH])
print('vqa_scoring.py 있음:', os.path.exists('/content/Expert-CFG/vqa_scoring.py'))
print('브랜치:', subprocess.run(['git', '-C', '/content/Expert-CFG', 'branch', '--show-current'], capture_output=True, text=True).stdout.strip())


In [ ]:
# 패키지 버전 체크 (baseline 노트북과 동일)
import importlib, subprocess, sys

def _ver(name):
    try: return importlib.metadata.version(name)
    except: return None

for _pkg, _min in [('torchao', (0,16))]:
    _v = _ver(_pkg)
    if _v is None:
        _install = {'open_clip_torch': 'open_clip_torch', 'faiss_gpu': 'faiss-gpu'}.get(_pkg, _pkg)
        print(f'{_pkg} 없음 → 설치 중...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _install])
    elif _pkg == 'torchao' and tuple(int(x) for x in _v.split('.')[:2]) < _min:
        print(f'torchao {_v} → 업그레이드 중...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torchao>=0.16.0'])
    else:
        print(f'{_pkg} {_v} OK')


In [ ]:
# 환경 3) 설정 — DATASET 을 바꿀 때는 런타임을 재시작하고 처음부터 한 번씩만 실행한다 (모델이 두 벌 올라가면 매우 느려진다)
DATASET = 'slake'          # 'vqarad' | 'slake'
KB      = 'peir,roco,trainqa'            # 검색 지식베이스: 'peir' | 'peir,roco,trainqa' (tools/build_rag_testjson.py --kb 와 같은 값. 파일이 먼저 만들어져 있어야 함)
KB_TAG  = '_v2'         # tools/build_rag_testjson.py --trainqa paper --retrieval union --out_suffix _v2 로 만든 통일 검색 파일(논문 · 팀원 방식). 이전 파일(자체 방식)은 ''
# 검색 변형 (4-2 절에서 같은 표로 비교): A union+QA = KB 'peir,roco,trainqa', KB_TAG '_v2' (현재)
#                                        B 이미지→텍스트+QA = 'peir,roco,trainqa', '_v2cross'  /  C PEIR+ROCO = 'peir,roco', '_v2'
# 변형마다 결과 폴더가 따로 생긴다 (ragcfg_{DATASET}_{KB}{KB_TAG}). 검색 파일은 4-2 절의 빌더 셀로 만든다.
RAG_SUFFIX = ('' if KB == 'peir' else '_' + KB.replace(',', '_')) + KB_TAG

E = '/content/drive/MyDrive/expert_cfg'
R = '/content/drive/MyDrive/ExpertCFG_replication'
DS = {
    'vqarad': dict(rag=f'{R}/rag_vqarad{RAG_SUFFIX}.json',
                   adapter=f'{E}/logs_phi35_vqarad_continue_e10_env_common/checkpoint-257',
                   baseline=f'{E}/vqarad_continue_e10_env_common_ckpt257_test.json',
                   img_root='/content/drive/MyDrive/vqa_rad/VQA_RAD Image Folder', n=451),
    'slake':  dict(rag=f'{R}/rag_slake{RAG_SUFFIX}.json',
                   adapter=f'{E}/logs_phi35_slake_continue_e5_env_common/checkpoint-380',
                   baseline=f'{E}/slake_continue_e5_env_common_ckpt380_test.json',
                   img_root='/content/slake_data/imgs', n=1061),
}
assert DATASET in DS

# 모델 (베이스라인 노트북과 같게)
BASE_MODEL_DIR  = './Phi-3.5-vision-instruct'
MED_LORA_HF     = 'ecoxial2007/Phi-3.5V-Med'
MED_LORA_DIR    = './lora_weights/logs_phi35_pubmed_instruct'
DOWNSTREAM_MODE = 'continue'          # 어댑터 하나를 base 에 얹는다 (Med LoRA 를 이어서 학습한 어댑터)
NUM_CROPS       = 16
MAX_NEW_TOKENS  = 64

# Expert-CFG 하이퍼파라미터
GUIDANCE_SCALE = 1.3     # 논문 Table 7 에서 γ∈{1, 1.3, 1.5} 중 1.3 이 최고 (본문 기본값 1.5, demo.py 기본값도 1.5)
ATTN_WEIGHT    = 3.0     # 논문 β=3, demo.py --attn (하이라이트 아닌 토큰)
PERTURB_WEIGHT = 0.01    # 논문 α=0.01 (하이라이트 토큰, unconditional 가지)
GATE_FRAC      = 0.05
GATE_ENTROPY   = 'eq4'    # 'eq4': 논문 식 4 (특수 토큰 <|end|> 제외 평균, 팀원 phi3_urp 와 같음) / 'stored': 저장된 베이스라인 결과의 entropy (<|end|> 포함)    # 엔트로피 상위 5%
import math
# 논문 식 7(임베딩 α) · 식 9/10(attention β, −δ) — 팀원 phi3_urp(phi3cfg/generation.py)와 같은 정의로 통일.
# 정상 가지: 하이라이트 key attention 에 +log β / 무조건부 가지: 하이라이트 임베딩 ×α 그리고 attention 에 −δ (δ = log β + 2)
PAPER_CFG      = dict(alpha=0.01, beta=3.0, delta=math.log(3.0) + 2.0)
RUN_RAG_ALL    = True    # 전 문항 w/ RAG (VQA-RAD 451 / SLAKE 1,061 생성). False 면 5% 행만
print('DATASET =', DATASET, '| KB =', KB, '| 검색 파일 접미어:', repr(RAG_SUFFIX))


In [ ]:
import os, sys, json, glob, re, random
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm   # 텍스트 진행 막대(위젯이 안 뜨는 환경에서도 보임)
from collections import Counter

# repo root
REPO_ROOT = Path('/content/Expert-CFG')
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import transformers, peft
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'

print('torch:', torch.__version__, '| transformers:', transformers.__version__)
print('GPU  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('BF16 :', USE_BF16)

# ── 무해한 경고 숨기기 (결과에는 영향 없음; 5.x ↔ 4.43 logits 비교로 확인) ──────────────
import warnings, logging
warnings.filterwarnings('ignore', message='.*attention mask API under.*')   # AttentionMaskConverter 사용 중단 예고

class _DropHarmless(logging.Filter):
    _pats = ('use_return_dict', 'You are not running the flash-attention implementation')
    def filter(self, record):
        msg = record.getMessage()
        return not any(p in msg for p in self._pats)

for _lg in (logging.getLogger(), logging.getLogger('transformers'), logging.getLogger('transformers_modules')):
    for _h in _lg.handlers:
        _h.addFilter(_DropHarmless())


## 1. 데이터 · 베이스라인

In [ ]:
# 데이터 · 베이스라인 짝짓기 (GPU 불필요)
import json, os, sys, numpy as np
from collections import defaultdict
sys.path.insert(0, '/content/Expert-CFG')
import vqa_scoring as vs

cfgd = DS[DATASET]
ADAPTER_DIR, IMG_ROOT = cfgd['adapter'], cfgd['img_root']
for p in (cfgd['rag'], cfgd['baseline'], f'{ADAPTER_DIR}/adapter_config.json'):
    assert os.path.exists(p), f'없음: {p}'
if DATASET == 'slake' and not os.path.isdir(IMG_ROOT):
    import zipfile
    from huggingface_hub import hf_hub_download
    with zipfile.ZipFile(hf_hub_download('BoKelvin/SLAKE', 'imgs.zip', repo_type='dataset')) as z:
        z.extractall('/content/slake_data')

rows = json.load(open(cfgd['rag'], encoding='utf-8'))          # 문항 + top_k_captions + highlights (tools/build_rag_testjson.py)
N = len(rows)
assert N == cfgd['n'], (N, cfgd['n'])
missing = [r['img_id'] for r in rows if not os.path.exists(os.path.join(IMG_ROOT, r['img_id']))]
assert not missing, f'이미지 누락 {len(missing)}개: {missing[:3]}'
test_anns = rows                                                 # 생성 함수 셀의 자체 점검이 이 이름을 쓴다
if 'top_k_sources' in rows[0]:
    from collections import Counter
    # 누수 진단: 검색된 top-1 캡션 안에 정답 · 질문이 이미 들어 있는가 (union 검색은 같은 이미지의 train QA 를 가져온다)
    _cap = lambda r: r['top_k_captions'][0].lower()
    _op = [r for r in rows if r['answer_type'] == 'OPEN']
    print('[누수 진단] 정답이 top-1 캡션에 그대로 있는 비율 (open):', f"{sum(str(r['answer']).strip().lower() in _cap(r) for r in _op) / max(1, len(_op)):.3f}",
          '| 질문 문장이 top-1 캡션에 그대로 있는 비율 (전체):', f"{sum(str(r['question']).strip().rstrip('?').strip().lower() in _cap(r) for r in rows) / len(rows):.3f}",
          '※ closed 의 yes/no 는 캡션에 흔한 단어라 정답 포함 비율을 재지 않는다')
    print('top-1 캡션 출처:', dict(Counter(r['top_k_sources'][0] for r in rows)), '| top-1 이 같은 이미지의 train QA 인 문항:', sum(r.get('top1_same_image', False) for r in rows), '(누수 가능성 — 있으면 해석 주의)')

base = json.load(open(cfgd['baseline'], encoding='utf-8'))

def pair_baseline(rows, base):
    nk = lambda s: str(s).strip().lower()
    same = lambda r, b: nk(r['question']) == nk(b['question']) and nk(r['answer']) == nk(b['answer'])
    if len(base) == len(rows) and all('idx' in b for b in base):                 # 1) 기록의 idx 가 rows 순서와 맞는가
        idxs = [b['idx'] for b in base]
        if sorted(idxs) == list(range(len(rows))) and all(same(rows[b['idx']], b) for b in base):
            return {b['idx']: b for b in base}, 'idx'
    if len(base) == len(rows) and all(same(r, b) for r, b in zip(rows, base)):   # 2) 같은 순서
        return {i: b for i, b in enumerate(base)}, '같은 순서'
    pool = defaultdict(list)                                                     # 3) (질문, 정답) 키
    for b in base: pool[(nk(b['question']), nk(b['answer']))].append(b)
    out = {}
    for i, r in enumerate(rows):
        k = (nk(r['question']), nk(r['answer']))
        assert pool[k], f'베이스라인에 없는 문항: {r["question"]!r}'
        out[i] = pool[k].pop(0)
    return out, '(질문, 정답) 키 — 이미지가 다른 중복 문항이 있으면 짝이 어긋날 수 있음'

base_by, how = pair_baseline(rows, base)
print(f'[{DATASET}] 문항 {N}개 (closed {sum(r["answer_type"] == "CLOSED" for r in rows)} / open {sum(r["answer_type"] == "OPEN" for r in rows)}) | 베이스라인 짝짓기: {how}')
assert all('entropy' in b and 'prediction' in b for b in base_by.values()), '베이스라인 결과에 entropy / prediction 이 없습니다'
base_preds = {i: base_by[i]['prediction'] for i in range(N)}
ent = np.array([base_by[i]['entropy'] for i in range(N)])
n_sel = int(round(GATE_FRAC * N))
sel = [int(i) for i in np.argsort(-ent)[:n_sel]]
print(f'엔트로피 상위 {GATE_FRAC*100:.0f}% = {n_sel}문항 (임계값 {ent[sel[-1]]:.3f}) | 그 중 closed {sum(rows[i]["answer_type"] == "CLOSED" for i in sel)}개')

def recs(preds):
    return [{'question': r['question'], 'answer': r['answer'], 'answer_type': r['answer_type'], 'prediction': preds[i]} for i, r in enumerate(rows)]
b0 = vs.summarize(recs(base_preds), vs.score_phi)
b1 = vs.summarize(recs(base_preds), vs.score_s1s2)
print('저장된 베이스라인 재채점 (open / closed / overall)')
print(f"  원본  (공식 채점): {b0['open']:.2f} / {b0['closed']:.2f} / {b0['overall']:.2f}")
print(f"  S1+S2 (참고, 규칙 완화): {b1['open']:.2f} / {b1['closed']:.2f} / {b1['overall']:.2f}")


## 2. 모델 로드와 생성 함수

아래 패치 · 로드 · 생성 셀은 `VQA_RAD_cfg_reproduction.ipynb` 의 것을 그대로 가져왔다 (transformers 5.x 호환 패치, 어텐션 bias 로 하이라이트 가중치를 주는 생성 루프).

In [ ]:
# ── transformers 5.x 호환: Phi3VForCausalLM _attn_implementation 패치 ──
_mfile = os.path.join(BASE_MODEL_DIR, 'modeling_phi3_v.py')
if os.path.exists(_mfile):
    with open(_mfile) as _f:
        _src = _f.read()
    _patched = _src
    for _old, _new in [
        ('def __init__(self, config: Phi3VConfig):',  'def __init__(self, config: Phi3VConfig, **kwargs):'),
        ('def __init__(self, config: PretrainedConfig):', 'def __init__(self, config: PretrainedConfig, **kwargs):'),
    ]:
        _patched = _patched.replace(_old, _new)
    if _patched == _src:
        _patched = re.sub(
            r'(def __init__\(self, config[^)]*?)(\):)',
            lambda m: m.group(0) if '**kwargs' in m.group(0) else m.group(1) + ', **kwargs' + m.group(2),
            _src
        )
    if _patched != _src:
        with open(_mfile, 'w') as _f:
            _f.write(_patched)
        print('modeling_phi3_v.py 패치 완료')
    else:
        print('이미 패치됨 / 패턴 미발견')
else:
    print(f'파일 없음: {_mfile}  (모델 다운로드 셀 먼저 실행)')


In [ ]:
import shutil
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoProcessor, GenerationConfig
from peft import PeftModel, LoraConfig, get_peft_model

_SKIP_FILES = {'processor_config.json', 'processing_phi3_v_cfg.py'}

# 1) 기본 가중치 다운로드
if not os.path.exists(os.path.join(BASE_MODEL_DIR, 'config.json')):
    print('Phi-3.5-vision-instruct 가중치 다운로드 중 (~8 GB)...')
    _hf_tmp = snapshot_download('microsoft/Phi-3.5-vision-instruct',
                                local_dir='/tmp/phi35_hf',
                                ignore_patterns=['*.git', '__pycache__'])
    os.makedirs(BASE_MODEL_DIR, exist_ok=True)
    for _f in os.listdir(_hf_tmp):
        if _f in _SKIP_FILES:
            continue
        _s, _d = os.path.join(_hf_tmp, _f), os.path.join(BASE_MODEL_DIR, _f)
        (shutil.copytree if os.path.isdir(_s) else shutil.copy2)(
            _s, _d, **({'dirs_exist_ok': True} if os.path.isdir(_s) else {}))
    print('완료')
else:
    print('기본 가중치 이미 존재')

with open(os.path.join(BASE_MODEL_DIR, 'processor_config.json')) as _f:
    assert 'processing_phi3_v_cfg' in _f.read(), \
        'processor_config.json 이 HF 원본으로 덮어써졌습니다 — repo 버전으로 복원하세요'

processor = AutoProcessor.from_pretrained(BASE_MODEL_DIR, trust_remote_code=True, num_crops=NUM_CROPS)

# 2) Med LoRA 다운로드/확인
def resolve_med_lora():
    if os.path.exists(os.path.join(MED_LORA_DIR, 'adapter_config.json')):
        return MED_LORA_DIR
    local = snapshot_download(MED_LORA_HF, local_dir='./lora_weights/Phi-3.5V-Med')
    found = glob.glob(os.path.join(local, '**', 'adapter_config.json'), recursive=True)
    assert found, f'{MED_LORA_HF} 에서 adapter_config.json 을 찾지 못했습니다'
    return os.path.dirname(found[0])

med_lora_path = resolve_med_lora()
print('Med LoRA :', med_lora_path)


In [ ]:
# ── Phi-3.5V + Med LoRA + VQA-RAD LoRA 로드 ─────────────────────────────
import glob as _glob
from transformers import AutoModelForCausalLM, AutoProcessor, GenerationConfig
from peft import PeftModel

processor = AutoProcessor.from_pretrained(
    BASE_MODEL_DIR, trust_remote_code=True, num_crops=NUM_CROPS)

# Med LoRA 경로 찾기
def resolve_med_lora():
    if os.path.exists(os.path.join(MED_LORA_DIR, 'adapter_config.json')):
        return MED_LORA_DIR
    from huggingface_hub import snapshot_download
    local = snapshot_download(MED_LORA_HF, local_dir='./lora_weights/Phi-3.5V-Med')
    found = _glob.glob(os.path.join(local, '**', 'adapter_config.json'), recursive=True)
    return os.path.dirname(found[0])

_dtype = torch.bfloat16 if USE_BF16 else torch.float16

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR, dtype=_dtype,
    trust_remote_code=True, _attn_implementation='eager')

if DOWNSTREAM_MODE == 'continue':
    # Med LoRA 를 이어서 학습한 어댑터 하나만 base 에 얹는다
    model = PeftModel.from_pretrained(base, ADAPTER_DIR).to(DEVICE).eval()
else:
    # Med LoRA merge 후 VQA-RAD LoRA 로드
    model = PeftModel.from_pretrained(base, resolve_med_lora()).merge_and_unload()
    model = PeftModel.from_pretrained(model, ADAPTER_DIR).to(DEVICE).eval()

try:
    model.generation_config = GenerationConfig.from_pretrained(BASE_MODEL_DIR)
except Exception:
    model.generation_config = GenerationConfig()

print('모델 로드 완료')


In [ ]:
# ── DynamicCache 패치 (transformers 5.x seen_tokens 복원) ────────────────
import importlib

_patched = []
for _mod_path in ['transformers.cache_utils', 'transformers.generation.cache_utils']:
    try:
        _mod = importlib.import_module(_mod_path)
        _DC  = getattr(_mod, 'DynamicCache', None)
        if _DC is None:
            continue
        _DC.seen_tokens = property(lambda self: self.get_seq_length())
        def _get_usable_length(self, new_seq_length, layer_idx=0):
            return self.get_seq_length(layer_idx)
        _DC.get_usable_length = _get_usable_length
        _DC.get_max_length    = lambda self: None

        # transformers 5.x 에서 제거된 legacy cache 변환 (모델 코드가 호출함)
        if not hasattr(_DC, 'from_legacy_cache'):
            def _from_legacy_cache(cls, past_key_values=None):
                cache = cls()
                if past_key_values is not None:
                    for layer_idx, (k, v) in enumerate(past_key_values):
                        cache.update(k, v, layer_idx)
                return cache
            _DC.from_legacy_cache = classmethod(_from_legacy_cache)
        if not hasattr(_DC, 'to_legacy_cache'):
            def _to_legacy_cache(self):
                if hasattr(self, 'layers'):
                    return tuple((l.keys, l.values) for l in self.layers)
                return tuple(zip(self.key_cache, self.value_cache))
            _DC.to_legacy_cache = _to_legacy_cache
        _patched.append(_mod_path)
    except Exception as _e:
        print(f'  {_mod_path} 패치 실패: {_e}')

print('DynamicCache 패치:', _patched)

# Phi3VForCausalLM 에 generate() 추가 (transformers 5.x 의 remote-code 모델은 GenerationMixin 이 빠져 있음)
from transformers import GenerationMixin
_cls = type(model.get_base_model())
if not issubclass(_cls, GenerationMixin):
    _cls.__bases__ = (GenerationMixin,) + _cls.__bases__
    print(f'Patched {_cls.__name__} with GenerationMixin')

# 검증
from transformers.cache_utils import DynamicCache as _DC2
_dc = _DC2(); print(f'seen_tokens = {_dc.seen_tokens}  (0 이면 정상)'); del _dc


In [ ]:
# ── Expert-CFG 생성 (repo demo.py 흐름 + soft attention 가중치) ────────────
#  - 프롬프트: "{top_k_captions[0]} <|image_1|>\n{question}"
#  - 같은 입력 2벌 → row0: 그대로, row1: 하이라이트 토큰만 attention 가중치를 낮춤
#  - ProbCFGLogitsProcessor(use_log=True) 로 두 가지를 대비
#
#  진단 결과: 이 모델 코드는 2D attention_mask 에서 1 이 아닌 값을 전부 "가림"으로 처리하므로
#  (attn=3,perturb=0.01) 을 그대로 넣으면 가중치가 아니라 row1 전체가 가려진다.
#  → 가중치는 각 self-attention 의 4D additive mask 에 log(weight) 로 더해 준다.
#    (softmax 전에 score + log(w) = 해당 key 의 attention 을 w 배 하는 것과 같다)
from src.highlighter_modules.guidance import ProbCFGLogitsProcessor

ATTN_BIAS = {'b': None}     # (2, L) float32: row0=0, row1=log(weight). None 이면 비활성
_BIAS_HOOKS = globals().get('_BIAS_HOOKS', [])

def _bias_pre_hook(module, args, kwargs):
    b = ATTN_BIAS['b']
    am = kwargs.get('attention_mask')
    if b is None or am is None or am.dim() != 4:
        return None
    if am.dtype == torch.bool:                       # bool mask → additive float mask
        zero = torch.zeros(am.shape, dtype=torch.float32, device=am.device)
        am = zero.masked_fill(~am, torch.finfo(torch.float32).min)
    kv = am.shape[-1]
    if b.dim() != 2 or am.shape[0] not in (1, b.shape[0]):
        raise RuntimeError(f'attention bias shape 불일치: mask={tuple(am.shape)}, bias={tuple(b.shape)}, '
                           f'layer={type(module).__name__}')
    bias = b[:, :kv]
    if bias.shape[-1] < kv:                          # 생성 단계에서 새로 붙는 토큰은 가중치 1 (bias 0)
        bias = torch.nn.functional.pad(bias, (0, kv - bias.shape[-1]))
    kwargs['attention_mask'] = am + bias[:, None, None, :].to(device=am.device, dtype=am.dtype)
    return args, kwargs

def install_attention_bias_hooks(m):
    for h in _BIAS_HOOKS:
        h.remove()
    _BIAS_HOOKS.clear()
    for name, mod in m.named_modules():
        if type(mod).__name__.endswith('Attention') and hasattr(mod, 'o_proj') and hasattr(mod, 'qkv_proj'):
            _BIAS_HOOKS.append(mod.register_forward_pre_hook(_bias_pre_hook, with_kwargs=True))
    print(f'attention bias hook 설치: {len(_BIAS_HOOKS)}개 레이어')

def build_expert_cfg_inputs(processor, image, question, caption, highlights,
                            attn=3.0, perturb=0.01, device='cuda', cond_boost=None):
    prompt = processor.tokenizer.apply_chat_template(
        [{'role': 'user', 'content': (f'{caption} ' if caption else '') + f'<|image_1|>\n{question}'}],
        tokenize=False, add_generation_prompt=True)
    inputs = processor(prompt, [image], return_tensors='pt',
                       qs_highlighted_parts=highlights).to(device)
    hl = inputs['highlight_attention_mask'].to(torch.float32)
    n_hl = int((hl == 1).sum())
    w = torch.where(hl == 1, torch.full_like(hl, perturb), torch.full_like(hl, attn))
    # cond_boost=β 이면 conditional 가지에서도 하이라이트 토큰의 attention 을 β 배 (논문 Expert-RAG 의 cond 가지)
    cond_bias = torch.zeros_like(w) if cond_boost is None else torch.where(
        hl == 1, torch.full_like(hl, float(np.log(cond_boost))), torch.zeros_like(hl))
    bias = torch.cat([cond_bias, torch.log(w.clamp_min(1e-6))], dim=0)   # (2, L)
    batch = {
        'input_ids':      inputs['input_ids'].repeat(2, 1),
        'pixel_values':   inputs['pixel_values'].repeat(2, 1, 1, 1, 1),
        'image_sizes':    inputs['image_sizes'].repeat(2, 1),
        'attention_mask': inputs['attention_mask'].repeat(2, 1),      # 일반 이진 mask
    }
    return batch, bias, n_hl

# ── 논문 식 7·9·10 (팀원 phi3_urp 의 phi3cfg/generation.py 와 같은 정의, 출처 표기) ──────────────
#   정상 가지(row0): 하이라이트 key 에 attention bias +log β
#   무조건부 가지(row1): 하이라이트 토큰 임베딩 ×α, 하이라이트 key 에 attention bias −δ
EMBED_SCALE = {'s': None}          # (2, L): row0=1, row1 = 1-(1-α)m. None 이면 비활성
_EMBED_HOOKS = globals().get('_EMBED_HOOKS', [])

def _embed_hook(module, args, output):
    sc = EMBED_SCALE['s']
    if sc is None or output.shape[1] < sc.shape[1]:        # 생성 단계(토큰 1개)에서는 건드리지 않는다
        return output
    out = output.clone()
    out[:, :sc.shape[1]] = out[:, :sc.shape[1]] * sc[:, :, None].to(device=out.device, dtype=out.dtype)
    return out

def install_embed_hook(m):
    for h in _EMBED_HOOKS:
        h.remove()
    _EMBED_HOOKS.clear()
    embs = [mod for n, mod in m.named_modules() if n.endswith('vision_embed_tokens')]
    assert len(embs) == 1, f'vision_embed_tokens 모듈이 {len(embs)}개입니다'
    _EMBED_HOOKS.append(embs[0].register_forward_hook(_embed_hook))
    print('embedding scale hook 설치: 1개')

def build_paper_cfg_inputs(processor, image, question, caption, highlights, alpha=0.01, beta=3.0, delta=None, device='cuda'):
    delta = float(np.log(beta) + 2.0) if delta is None else float(delta)
    prompt = processor.tokenizer.apply_chat_template(
        [{'role': 'user', 'content': (f'{caption} ' if caption else '') + f'<|image_1|>\n{question}'}],
        tokenize=False, add_generation_prompt=True)
    inputs = processor(prompt, [image], return_tensors='pt', qs_highlighted_parts=highlights).to(device)
    m = (inputs['highlight_attention_mask'].to(torch.float32) == 1).float()[0]        # (L,) 하이라이트 토큰 = 1
    n_hl = int(m.sum())
    bias = torch.stack([float(np.log(beta)) * m, -delta * m])                         # (2, L)
    scale = torch.stack([torch.ones_like(m), 1.0 - (1.0 - alpha) * m])                # (2, L)
    batch = {
        'input_ids':      inputs['input_ids'].repeat(2, 1),
        'pixel_values':   inputs['pixel_values'].repeat(2, 1, 1, 1, 1),
        'image_sizes':    inputs['image_sizes'].repeat(2, 1),
        'attention_mask': inputs['attention_mask'].repeat(2, 1),
    }
    return batch, bias, scale, n_hl

@torch.no_grad()
def generate_expert_cfg(model, processor, image, question, caption, highlights,
                        cfg=1.5, attn=3.0, perturb=0.01, max_new_tokens=64, device='cuda', cond_boost=None, paper=None):
    """model.generate() 대신 forward 만 쓰는 greedy 루프.
    (transformers 5.x + 구버전 remote code 에서 generate 내부의 position_ids/cache 처리가 어긋남)
    ProbCFGLogitsProcessor(use_log=True) 와 같은 계산: guided = cfg*(cond-uncond)+uncond (log-prob 기준)."""
    model.eval()
    tok = processor.tokenizer
    eos_ids = {tok.eos_token_id, tok.convert_tokens_to_ids('<|end|>')}
    if paper is not None:      # 논문 식 7·9·10 (통일 정의). paper = dict(alpha, beta, delta)
        batch, bias, scale, n_hl = build_paper_cfg_inputs(processor, image, question, caption, highlights, device=device, **paper)
        EMBED_SCALE['s'] = scale
    else:                      # 이전 자체 정의 (비교용으로만 남김)
        batch, bias, n_hl = build_expert_cfg_inputs(processor, image, question, caption, highlights,
                                                    attn, perturb, device, cond_boost)
    L = batch['input_ids'].shape[1]
    ATTN_BIAS['b'] = bias
    new_ids = []
    try:
        out = model(**batch, use_cache=True)                      # prefill (이미지 포함)
        past = out.past_key_values
        logits = out.logits[:, -1].float()
        for step in range(max_new_tokens):
            lp = torch.log_softmax(logits, dim=-1)                # (2, V): row0=cond, row1=uncond
            guided = cfg * (lp[0] - lp[1]) + lp[1]
            next_id = int(guided.argmax())
            if next_id in eos_ids:
                break
            new_ids.append(next_id)
            nxt = torch.full((2, 1), next_id, dtype=torch.long, device=device)
            am = torch.ones((2, L + step + 1), dtype=torch.long, device=device)
            pos = torch.full((2, 1), L + step, dtype=torch.long, device=device)
            out = model(input_ids=nxt, attention_mask=am, position_ids=pos,
                        past_key_values=past, use_cache=True)
            past = out.past_key_values
            logits = out.logits[:, -1].float()
    finally:
        ATTN_BIAS['b'] = None
        EMBED_SCALE['s'] = None
    return tok.decode(new_ids, skip_special_tokens=True).strip().strip('.'), n_hl

@torch.no_grad()
def check_mask_effect(ann, settings=((1.0, 1.0), (3.0, 3.0), (1.0, 0.5), (1.0, 0.01), (3.0, 0.01), (None, None))):
    """row0/row1 마지막 logits 차이.
    기대: (1,1)=(3,3)=0 (가중치가 균일), (1,0.5) 는 0 이 아니고 (1,0.01) 보다 작음,
          (3,0.01) ≈ (1,0.0033) (가중치는 비율만 중요)."""
    image = Image.open(os.path.join(IMG_ROOT, ann['img_id'])).convert('RGB')
    for a, pw in settings:
        a = ATTN_WEIGHT if a is None else a
        pw = PERTURB_WEIGHT if pw is None else pw
        batch, bias, n_hl = build_expert_cfg_inputs(processor, image, ann['question'],
                                                    ann['top_k_captions'][0], ann['highlights'],
                                                    a, pw, DEVICE)
        ATTN_BIAS['b'] = bias
        try:
            logits = model(**batch, use_cache=False).logits[:, -1].float()
        finally:
            ATTN_BIAS['b'] = None
        diff = (logits[0] - logits[1]).abs().max().item()
        print(f'attn={a}, perturb={pw} | 하이라이트 토큰={n_hl} | 마지막 logits 최대 차이={diff:.4f}')

@torch.no_grad()
def check_paper_cfg(ann):
    """팀원 tests/cpu_smoke.py 와 같은 점검: 중립 설정(α=1, β=1, δ=0)이면 두 가지의 logits 가 같고, 논문 설정이면 달라진다."""
    image = Image.open(os.path.join(IMG_ROOT, ann['img_id'])).convert('RGB')
    for name, kw in (('중립 (α=1, β=1, δ=0)', dict(alpha=1.0, beta=1.0, delta=0.0)), ('논문 (α=0.01, β=3, δ=log3+2)', dict(alpha=0.01, beta=3.0))):
        batch, bias, scale, n_hl = build_paper_cfg_inputs(processor, image, ann['question'], ann['top_k_captions'][0], ann['highlights'], device=DEVICE, **kw)
        ATTN_BIAS['b'] = bias; EMBED_SCALE['s'] = scale
        try:
            logits = model(**batch, use_cache=False).logits[:, -1].float()
        finally:
            ATTN_BIAS['b'] = None; EMBED_SCALE['s'] = None
        print(f'{name}: 하이라이트 토큰={n_hl} | 두 가지의 마지막 logits 최대 차이={(logits[0] - logits[1]).abs().max().item():.4f}')

install_attention_bias_hooks(model)
install_embed_hook(model)
print('Expert-CFG 생성 함수 정의 완료')

def find_ann_with_highlight(anns, limit=300):
    """하이라이트 토큰이 실제로 잡히는 첫 문항 (점검이 의미를 가지려면 하이라이트 토큰이 1개 이상이어야 한다)."""
    for k, a in enumerate(anns[:limit]):
        img = Image.open(os.path.join(IMG_ROOT, a['img_id'])).convert('RGB')
        n = build_paper_cfg_inputs(processor, img, a['question'], a['top_k_captions'][0], a['highlights'], device=DEVICE)[3]
        if n > 0:
            print(f'점검 문항: #{k} (하이라이트 토큰 {n}개) | 하이라이트 = {a["highlights"][:3]}')
            return a
    print('경고: 앞 문항들에서 하이라이트 토큰이 잡히는 문항을 찾지 못했습니다 — 하이라이트 문자열이 프롬프트에서 매칭되는지 확인하세요')
    return anns[0]
_ann_hl = find_ann_with_highlight(test_anns)
check_mask_effect(_ann_hl)
check_paper_cfg(_ann_hl)        # 기대: 중립이면 0 에 가깝고(bf16 잡음 정도), 논문 설정이면 0 보다 확실히 큼
_a = _ann_hl
_pred, _ = generate_expert_cfg(model, processor, Image.open(os.path.join(IMG_ROOT, _a['img_id'])).convert('RGB'),
                               _a['question'], _a['top_k_captions'][0], _a['highlights'],
                               cfg=GUIDANCE_SCALE, max_new_tokens=MAX_NEW_TOKENS, device=DEVICE, paper=PAPER_CFG)
print('샘플 생성 테스트 →', repr(_pred), '| GT:', _a['answer'])


In [ ]:
# 게이팅 엔트로피 — 논문 식 4 (팀원 phi3_urp 와 같은 정의): 답의 각 토큰 위치에서 어휘 전체의 엔트로피를 구해, 특수 토큰(<|end|> 등)을 뺀 토큰들로 평균
# 저장된 베이스라인 결과의 entropy 는 <|end|> 위치까지 평균에 들어 있다(closed 답은 yes + <|end|> 두 토큰이라 영향이 큼).
# 저장된 베이스라인 예측(greedy)을 그대로 teacher-forcing 해서 구한다 — 재토큰화 때문에 생성 때의 토큰과 드물게 다를 수 있다. 문항당 forward 1번.
import json, os, numpy as np, torch
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

ent_stored = np.array(ent, dtype=float).copy()
if GATE_ENTROPY == 'eq4':
    tok = processor.tokenizer
    special = set(tok.all_special_ids)
    EQ4_CACHE = f'{E}/ent_eq4_{DATASET}.json'
    eq4 = {int(k): v for k, v in json.load(open(EQ4_CACHE)).items()} if os.path.exists(EQ4_CACHE) else {}

    @torch.no_grad()
    def eq4_entropy(r, pred):
        image = Image.open(os.path.join(IMG_ROOT, r['img_id'])).convert('RGB')
        prompt = tok.apply_chat_template([{'role': 'user', 'content': f"<|image_1|>\n{r['question']}"}], tokenize=False, add_generation_prompt=True)
        inp = processor(prompt, [image], return_tensors='pt', qs_highlighted_parts=[])
        inp.pop('highlight_attention_mask', None)
        inp = {k: v.to(DEVICE) for k, v in inp.items()}
        ans = tok(pred, add_special_tokens=False, return_tensors='pt')['input_ids'].to(DEVICE)
        if ans.shape[1] == 0:
            return None
        L = inp['input_ids'].shape[1]
        ids = torch.cat([inp['input_ids'], ans], dim=1)
        inp2 = dict(inp); inp2['input_ids'] = ids; inp2['attention_mask'] = torch.ones_like(ids)
        lp = torch.log_softmax(model(**inp2, use_cache=False).logits[0, L - 1:-1].float(), dim=-1)      # 답 토큰을 예측하는 위치들
        h = (-(lp.exp() * lp).sum(-1)).cpu().numpy()
        keep = [j for j, t in enumerate(ans[0].tolist()) if t not in special]
        return float(h[keep].mean()) if keep else None

    model.eval()
    for i in tqdm([i for i in range(N) if i not in eq4], desc='식 4 엔트로피'):
        v = eq4_entropy(rows[i], base_preds[i])
        eq4[i] = -1.0 if v is None else v
        if len(eq4) % 100 == 0: json.dump({str(k): v for k, v in eq4.items()}, open(EQ4_CACHE, 'w'))      # 100문항마다 중간 저장 (끊겨도 이어서 계산)
    json.dump({str(k): v for k, v in eq4.items()}, open(EQ4_CACHE, 'w'))
    ent = np.array([eq4[i] for i in range(N)], dtype=float)
    ent[ent < 0] = ent[ent >= 0].min() if (ent >= 0).any() else 0.0        # 빈 답은 가장 확실한 쪽으로
    sel_old = set(int(i) for i in np.argsort(-ent_stored)[:n_sel])
    sel = [int(i) for i in np.argsort(-ent)[:n_sel]]
    print(f'게이팅: 식 4 엔트로피 상위 {n_sel}문항 (임계값 {ent[sel[-1]]:.3f}) | 저장된 entropy 기준 선택과 겹침 {len(sel_old & set(sel))}/{n_sel}')
    s_ = np.array([vs.score_phi(r) for r in recs(base_preds)]); wrong = (s_ < 1.0).astype(int)
    for nm, v in (('저장된 entropy(<|end|> 포함)', ent_stored), ('식 4(특수 토큰 제외)', ent)):
        print(f'  {nm:26s} 전체 AUC {roc_auc_score(wrong, v):.3f} | closed {roc_auc_score(wrong[[r["answer_type"] == "CLOSED" for r in rows]], v[[r["answer_type"] == "CLOSED" for r in rows]]):.3f} | 선택 {n_sel}문항의 오답률 {wrong[np.argsort(-v)[:n_sel]].mean():.3f}')
else:
    print('게이팅: 저장된 베이스라인 entropy 사용 (GATE_ENTROPY="stored")')
print('선택 문항 수:', len(sel), '| closed', sum(rows[i]['answer_type'] == 'CLOSED' for i in sel))


## 3. 실험 (GPU)

In [ ]:
# 실험 실행 — 결과는 Drive 에 문항 단위로 저장되고, 끊겨도 다시 실행하면 이어서 계산한다
import time
from tqdm import tqdm
OUT = f'{E}/ragcfg_{DATASET}{RAG_SUFFIX}' + ('' if GATE_ENTROPY == 'eq4' else '_gate_stored')          # KB · 게이팅 정의별로 폴더를 나눠 캐시가 섞이지 않게 한다 (stored 는 _gate_stored 폴더, 보고용이 아닌 빠른 점검)
os.makedirs(OUT, exist_ok=True)

def oracle_highlights(r):
    """(현재 EXPS 에서 쓰지 않음) open 문항은 정답 자체를 하이라이트하는 일반 oracle 행. 정답이 캡션에 없으면 마스크가 생기지 않아 행에서 뺐다."""
    if r['answer_type'] == 'OPEN':
        a = r['answer'].strip()
        return list(dict.fromkeys([a] + a.split()))
    return r['highlights']

# 팀원 phi3_urp(retrieval.py keyword_highlights) 규칙: closed 문항은 질문과 내용어가 가장 많이 겹치는 검색 캡션 문장(최대 2개)을 하이라이트.
# 겹치는 문장이 없으면 하이라이트 없음 → 가이던스 없이(= RAG 와 같은 입력) 생성. open 문항의 규칙 준수 행은 기존 규칙(자체 키워드)을 그대로 쓴다.
_STOP = set("""a an the is are was were be been of in on at to for from by with and or not no yes this that these those there here it its their
what which where when how why who whom does do did can could would should shown show shows seen see image images picture photo scan visible present
any some into than as about type kind part side""".split())
_words = lambda t: re.findall(r'[a-z0-9]+(?:-[a-z0-9]+)*', str(t).lower())
_content = lambda t: {w for w in _words(t) if w not in _STOP and len(w) > 1}
def closed_sentence_highlights(question, caption):
    sents = [x for x in re.split(r'(?<=[.!?])\s+', caption) if x.strip()]
    q = _content(question)
    scored = [(len(q & _content(x)), x) for x in sents]
    best = max((n for n, _ in scored), default=0)
    return [x.strip() for n, x in scored if n == best and n > 0][:2]
def compliant_highlights(r):
    """규칙 준수 행의 하이라이트: closed = 질문과 가장 많이 겹치는 캡션 문장(최대 2개, 팀원 규칙), open = 검색 파일의 규칙 기반 하이라이트. 정답은 쓰지 않는다. (옛 이름 peir_highlights 는 PEIR 와 무관해서 바꿨다)"""
    return closed_sentence_highlights(r['question'], r['top_k_captions'][0]) if r['answer_type'] == 'CLOSED' else r['highlights']

# 키 'peir' 는 결과 파일 · 출력 이름과의 호환 때문에 유지한다 (규칙 준수 하이라이트를 뜻함)
HL = {'none': lambda r: [], 'peir': lambda r: compliant_highlights(r), 'oracle': oracle_highlights, 'inject': lambda r: inject_highlights(r)}

def inject_caption(r):
    """논문 · 팀원(phi3_urp) 방식: open 문항은 정답을 전문가 주석으로 쓴다. 검색 캡션에 정답이 없으면 캡션 뒤에 덧붙인다. closed 는 그대로."""
    cap = r['top_k_captions'][0]
    if r['answer_type'] != 'OPEN':
        return cap
    gt = r['answer'].strip()
    return cap if gt.lower() in cap.lower() else f'{cap} {gt}.'

def inject_highlights(r):
    """정답 주입 행: open = 정답 문자열, closed = 검색 파일의 highlights(n-gram). 주의: 규칙 준수 행의 closed(문장 겹침 규칙)와 다르다. common/highlights.py 의 inject_highlights 는 closed 대체값을 문장 규칙으로 받는다."""
    return [r['answer'].strip()] if r['answer_type'] == 'OPEN' else r['highlights']

CAP = {k: (lambda r: r['top_k_captions'][0]) for k in ('none', 'peir', 'oracle')}
CAP['inject'] = inject_caption

def run(name, idxs, hl, cfg, cond_boost, paper=None):
    f = f'{OUT}/{name}.json'
    preds = {int(k): v for k, v in json.load(open(f, encoding='utf-8')).items()} if os.path.exists(f) else {}
    todo = [i for i in idxs if i not in preds]
    if todo: print(f'{name}: {len(idxs)}문항 중 {len(todo)}개 생성')
    save = lambda: json.dump({str(k): v for k, v in preds.items()}, open(f, 'w', encoding='utf-8'), ensure_ascii=False)
    t0 = time.time()
    for n, i in enumerate(tqdm(todo, desc=name, leave=False)):
        r = rows[i]
        image = Image.open(os.path.join(IMG_ROOT, r['img_id'])).convert('RGB')
        pred, _ = generate_expert_cfg(model, processor, image, r['question'], CAP[hl](r), HL[hl](r),
                                      cfg=cfg, attn=ATTN_WEIGHT, perturb=PERTURB_WEIGHT,
                                      max_new_tokens=MAX_NEW_TOKENS, device=DEVICE, cond_boost=cond_boost, paper=paper)
        preds[i] = pred
        if (n + 1) % 25 == 0: save()
    if todo: save(); print(f'  완료 ({(time.time() - t0) / 60:.1f}분)')
    return {i: preds[i] for i in idxs}

# (이름, 대상 문항, 하이라이트, cfg, cond_boost, 전체/게이팅, paper)
#  - 통일 정의: CFG 는 논문 식 7·9·10 (PAPER_CFG, 팀원 phi3_urp 와 같음). Expert-RAG 는 가이던스 없이 전문가 참조만 프롬프트에 넣는다.
#  - 정답을 쓰지 않는 규칙 준수 행에서는 Expert-RAG 가 5% RAG 와 같다(참조 = 같은 top-1 캡션) → 별도 행을 두지 않는다.
#  - 이름에 -v2 : 이전(자체 정의) 결과 캐시와 섞이지 않게 하려는 표시
EXPS = [
    # 기본 비교 행 = 논문 방식(정답 주입, 규칙 완화)을 먼저, 규칙 준수 행은 그 다음. 전 문항 RAG-100 은 가장 오래 걸려서(VQA-RAD 약 33분, SLAKE 약 70분) 마지막에 둔다.
    # γ: 논문 본문 1.5 / 부록 Table 7 최고 1.3 (논문 안에서 값이 다름) → 두 값을 모두 보고
    ('RAG-5',                 sel,            'none',   1.0,            None, 'gate', None),
    ('ExpertRAG-5-inject-v2', sel,            'inject', 1.0,            None, 'gate', None),
    ('ExpertCFG-5-inject-v2', sel,            'inject', GUIDANCE_SCALE, None, 'gate', PAPER_CFG),
    ('ExpertCFG-5-inject-v2-g15', sel,        'inject', 1.5,            None, 'gate', PAPER_CFG),
    ('ExpertCFG-5-v2',        sel,            'peir',   GUIDANCE_SCALE, None, 'gate', PAPER_CFG),
    ('ExpertCFG-5-v2-g15',        sel,        'peir',   1.5,            None, 'gate', PAPER_CFG),
    ('RAG-100',               list(range(N)), 'none',   1.0,            None, 'all',  None),
]
def highlight_coverage(hlkey):
    """선택 문항 중 하이라이트 토큰이 실제로 잡히는 문항 수 (0 이면 CFG 가 RAG 와 같은 입력이 된다)."""
    ns = []
    for i in sel:
        r = rows[i]
        img = Image.open(os.path.join(IMG_ROOT, r['img_id'])).convert('RGB')
        ns.append(build_paper_cfg_inputs(processor, img, r['question'], CAP[hlkey](r), HL[hlkey](r), device=DEVICE)[3])
    nz = [n for n in ns if n > 0]
    print(f"  하이라이트 '{hlkey}': 토큰이 있는 문항 {len(nz)}/{len(sel)} | 있는 문항의 평균 {np.mean(nz) if nz else 0:.1f}토큰 (최대 {max(ns) if ns else 0})"
          f" | closed {sum(n > 0 for n, i in zip(ns, sel) if rows[i]['answer_type'] == 'CLOSED')}/{sum(rows[i]['answer_type'] == 'CLOSED' for i in sel)}, open {sum(n > 0 for n, i in zip(ns, sel) if rows[i]['answer_type'] == 'OPEN')}/{sum(rows[i]['answer_type'] == 'OPEN' for i in sel)}")
print('[하이라이트 적용 점검] 선택', len(sel), '문항')
for _k in ('peir', 'inject'):
    highlight_coverage(_k)

EXP_PREDS = {}
for name, idxs, hl, cfg, cb, kind, pp in EXPS:
    if name == 'RAG-100' and not RUN_RAG_ALL:
        continue
    p = run(name, idxs, hl, cfg, cb, pp)
    EXP_PREDS[name] = p if kind == 'all' else {**base_preds, **p}      # 게이팅 행: 선택 문항만 바꾸고 나머지는 베이스라인 예측
print('\n실험 완료:', list(EXP_PREDS))


## 4. 결과

In [ ]:
# 결과표 — 원본 채점(공식) / S1+S2 (참고) · 논문 대비 · 베이스라인 대비(문항 쌍 부트스트랩 95% CI)
PAPER = {
    'vqarad': {'baseline': (69.27, 90.44, 82.04), 'rag': (70.39, 88.24, 81.15), 'rag5': (70.95, 89.34, 82.04), 'expert_rag': (72.06, 90.44, 83.37), 'expert_cfg': (74.86, 91.18, 84.70)},
    'slake':  {'baseline': (82.95, 88.70, 85.20), 'rag': (82.79, 89.90, 85.58), 'rag5': (82.64, 89.42, 85.30), 'expert_rag': (83.56, 89.90, 86.05), 'expert_cfg': (85.27, 91.59, 87.75)},
}[DATASET]
META = {  # 행 이름: (표시 이름, 논문 행, 규칙)
    'baseline':              ('베이스라인',                          'baseline',   '규칙 준수'),
    'RAG-100':               ('w/ RAG (top-1, 전 문항)',             'rag',        '자체 검색'),
    'RAG-5':                 ('w/ 5% RAG',                          'rag5',       '자체 검색'),
    'ExpertCFG-5-v2':        ('w/ 5% Expert-CFG (논문 식 7~10)',     'expert_cfg', '하이라이트: open=검색 파일 규칙 / closed=질문과 겹치는 캡션 문장, 정답 미사용'),
    'ExpertRAG-5-inject-v2': ('w/ 5% Expert-RAG (논문 방식: 전문가 주석=정답)', 'expert_rag', '규칙 완화(논문 방식 모사): open 참조에 정답 덧붙임'),
    'ExpertCFG-5-inject-v2': ('w/ 5% Expert-CFG (논문 방식: 전문가 주석=정답)', 'expert_cfg', '규칙 완화(논문 방식 모사): open 참조에 정답 덧붙임, 정답 하이라이트'),
    'ExpertCFG-5-v2-g15':        ('w/ 5% Expert-CFG (γ=1.5)',          'expert_cfg', '하이라이트: open=검색 파일 규칙 / closed=질문과 겹치는 캡션 문장, 정답 미사용'),
    'ExpertCFG-5-inject-v2-g15': ('w/ 5% Expert-CFG (논문 방식, γ=1.5)', 'expert_cfg', '규칙 완화(논문 방식 모사)'),
}
ALL_PREDS = {'baseline': base_preds, **EXP_PREDS}
rng = np.random.default_rng(0)
def paired_ci(new, ref, n=10000):
    d = new - ref
    return np.percentile(d[rng.integers(0, len(d), (n, len(d)))].mean(1) * 100, [2.5, 97.5])

import pandas as pd
from IPython.display import display
SCORERS = (('원본', vs.score_phi), ('S1+S2', vs.score_s1s2))
base_arr = np.array([vs.score_phi(r) for r in recs(base_preds)])
summary, main_rows, relaxed_rows, delta_rows = {}, [], [], []
for key, preds in ALL_PREDS.items():
    label, pk, rule = META[key]
    pap = PAPER[pk]
    rec = recs(preds)
    m = {lab: vs.summarize(rec, sc_) for lab, sc_ in SCORERS}
    relaxed = '-inject' in key
    row = {'행': label, 'open': m['원본']['open'], 'closed': m['원본']['closed'], 'overall': m['원본']['overall'],
           'S1+S2 overall(참고)': m['S1+S2']['overall'], '논문 overall': pap[2], '논문 대비': m['원본']['overall'] - pap[2]}
    (relaxed_rows if relaxed else main_rows).append(row)
    if key != 'baseline':
        arr = np.array([vs.score_phi(r) for r in rec])
        lo, hi = paired_ci(arr, base_arr)
        delta_rows.append({'행': label, '구분': '규칙 완화' if relaxed else '규칙 준수',
                           '우리 Δ': f"{(arr - base_arr).mean() * 100:+.2f}  [{lo:+.2f}, {hi:+.2f}]",
                           '논문 Δ': f"{pap[2] - PAPER['baseline'][2]:+.2f}"})
    for lab in m:
        summary.setdefault(key, {})[lab] = {k: float(m[lab][k]) for k in ('open', 'closed', 'overall')}
    summary[key]['paper'] = PAPER[pk]; summary[key]['rule'] = rule

def show(rows, title, num_cols=()):
    print(title)
    st = pd.DataFrame(rows).style.format({c_: '{:.2f}' for c_ in num_cols if c_ not in ('논문 대비',)} | ({'논문 대비': '{:+.2f}'} if '논문 대비' in num_cols else {}))
    try: st = st.hide(axis='index')
    except Exception: pass
    display(st.set_properties(**{'text-align': 'right'}).set_properties(subset=['행'] + (['구분'] if rows and '구분' in rows[0] else []), **{'text-align': 'left'}))

NUM = ('open', 'closed', 'overall', 'S1+S2 overall(참고)', '논문 overall', '논문 대비')
print(f'[{DATASET}] N={N}  KB={KB}   (논문: Phi3.5V-Med)')
if relaxed_rows:
    show(relaxed_rows, '\n■ 논문 방식 행 (기본 비교) — 전문가 주석 = 정답을 참조에 덧붙임. 규칙 완화(oracle): test 정답이 입력에 들어가므로 규칙 준수 결과로 보고하지 않는다', NUM)
show(main_rows, '\n■ 규칙 준수 행 (정답 미사용; 논문 방식 행과 입력 정보가 다르다) — open / closed / overall = 공식 채점(원본, Phi 3.5V 방식), S1+S2 overall 은 참고', NUM)
delta_rows.sort(key=lambda d_: d_['구분'] != '규칙 완화')      # 논문 방식 행이 먼저
show(delta_rows, '\n■ 베이스라인 대비 overall 변화 (원본 채점, 우리는 문항 쌍 부트스트랩 95% CI)')
json.dump({'dataset': DATASET, 'n': N, 'kb': KB, 'rag_file': cfgd['rag'], 'gate': GATE_ENTROPY, 'n_selected': len(sel), 'rows': summary}, open(f'{OUT}/summary.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('\n※ 공식 채점 = 원본(Phi 3.5V 방식). S1+S2 는 Phi 의 VQA-RAD test 오답에서 도출된 규칙이라 규칙 완화 참고값으로만 쓴다.')
print('저장:', f'{OUT}/summary.json', '| 게이팅 정의:', GATE_ENTROPY)


In [ ]:
# 불확실성(평균 토큰 엔트로피)이 오답을 가려내는가 — ROC-AUC (GPU 불필요, 저장된 베이스라인 결과만 사용)
from sklearn.metrics import roc_auc_score
rec = recs(base_preds); s = np.array([vs.score_phi(r) for r in rec]); wrong = (s < 1.0).astype(int)
isc = np.array([r['answer_type'] == 'CLOSED' for r in rec])
print(f'[{DATASET}] 오답(점수<1) 비율 {wrong.mean():.3f}')
for name, m in (('전체', np.ones(N, bool)), ('closed', isc), ('open', ~isc)):
    if wrong[m].min() == wrong[m].max(): print(f'  {name}: 한 클래스뿐이라 AUC 정의 불가'); continue
    print(f'  {name:7s} AUC = {roc_auc_score(wrong[m], ent[m]):.3f}  (n={int(m.sum())})')
print('상위 5% 선택 문항의 오답률:', f'{wrong[sel].mean():.3f}', '| 전체 오답률:', f'{wrong.mean():.3f}')


In [ ]:
# AUC 정의 비교 (GPU 필요, 문항당 forward 1번) — closed AUC 가 논문(0.958)보다 낮은 이유 확인
# 저장된 베이스라인 예측을 그대로 teacher-forcing 해서 답 토큰 위치별 엔트로피를 구한다 (greedy 답이므로 생성 때의 scores 와 같다).
# 비교 항목: (a) 저장된 값  (b) 전체 평균(<|end|> 포함)  (c) 전체 평균(<|end|> 제외)  (d) 첫 토큰  (e) 최대
# 오답 기준도 두 가지로: 원본(Phi 식) · S1+S2
import torch, numpy as np
from PIL import Image
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

tok = processor.tokenizer
end_id = tok.convert_tokens_to_ids('<|end|>')
CLOSED_ONLY = True      # False 면 open 도 계산 (시간 3배)

@torch.no_grad()
def token_entropies(r, pred):
    image = Image.open(os.path.join(IMG_ROOT, r['img_id'])).convert('RGB')
    prompt = tok.apply_chat_template([{'role': 'user', 'content': f"<|image_1|>\n{r['question']}"}],
                                     tokenize=False, add_generation_prompt=True)
    inp = processor(prompt, [image], return_tensors='pt', qs_highlighted_parts=[])
    inp.pop('highlight_attention_mask', None)
    inp = {k: v.to(DEVICE) for k, v in inp.items()}
    ans = tok(pred, add_special_tokens=False, return_tensors='pt')['input_ids'].to(DEVICE)
    ans = torch.cat([ans, torch.tensor([[end_id]], device=DEVICE)], dim=1)       # 생성은 <|end|> 에서 멈춘다
    L = inp['input_ids'].shape[1]
    ids = torch.cat([inp['input_ids'], ans], dim=1)
    inp2 = dict(inp); inp2['input_ids'] = ids
    inp2['attention_mask'] = torch.ones_like(ids)
    logits = model(**inp2, use_cache=False).logits[0, L - 1:-1].float()           # 답 토큰(+end)을 예측하는 위치들
    lp = torch.log_softmax(logits, -1)
    return (-(lp.exp() * lp).sum(-1)).cpu().numpy()                               # (답 토큰 수 + 1,)

model.eval()
todo = [i for i in range(N) if (not CLOSED_ONLY) or rows[i]['answer_type'] == 'CLOSED']
TE = {}
for i in tqdm(todo, desc='token entropy'):
    TE[i] = token_entropies(rows[i], base_preds[i])

rec = recs(base_preds)
wr = {'원본': np.array([vs.score_phi(r) < 1.0 for r in rec]).astype(int),
      'S1+S2': np.array([vs.score_s1s2(r) < 1.0 for r in rec]).astype(int)}
variants = {
    '저장된 값':          lambda i: ent[i],
    '전체 평균(end 포함)': lambda i: TE[i].mean(),
    '전체 평균(end 제외)': lambda i: TE[i][:-1].mean() if len(TE[i]) > 1 else TE[i][0],
    '첫 토큰':            lambda i: TE[i][0],
    '최대':               lambda i: TE[i].max(),
}
print(f'[{DATASET}] closed {len(todo)}문항 | 저장된 값과 (b) 의 상관: {np.corrcoef([ent[i] for i in todo], [TE[i].mean() for i in todo])[0,1]:.3f} (1 에 가까워야 재현이 맞음)')
print(f"{'엔트로피 정의':<18}" + ''.join(f'{k + " AUC":>14}' for k in wr))
for name, f in variants.items():
    v = np.array([f(i) for i in todo])
    out = []
    for k, w in wr.items():
        y = w[todo]
        out.append(f'{roc_auc_score(y, v):14.3f}' if y.min() != y.max() else f'{"정의불가":>14}')
    print(f'{name:<18}' + ''.join(out))
print('오답률(closed):', {k: round(float(w[todo].mean()), 3) for k, w in wr.items()}, '| 논문 Table 6 closed AUC (파인튜닝): VQA-RAD 0.958 / SLAKE 0.949')


## 4-2. RAG 진단 · 검색 방식 비교 (GPU 불필요)

3절에서 저장한 결과(`ragcfg_*`)와 검색 파일만 읽는다. 두 가지를 본다.

1. **RAG 하락 진단** — w/ RAG(전 문항)가 베이스라인보다 낮아지는 이유 (VQA-RAD −4.83, 논문 −0.89). open / closed 분해, 정답→오답으로 뒤집힌 문항이 어떤 검색 조건에 몰리는지, closed 의 yes 편향, 가장 나빠진 예를 보여 준다. **원인은 단정하지 않는다.**
2. **검색 방식 비교** — 같은 표에서 A(union, 팀원 방식) / B(이미지→텍스트만, 이미지별로 합친 train QA 포함) / C(PEIR+ROCO만, QA 없음). 점수는 모두 공식(원본 Phi 3.5V) 채점이고, Δ는 문항 쌍 부트스트랩 95% CI와 함께 본다.

변형을 돌리는 순서: 아래 빌더 셀로 B·C 검색 파일을 만든다(한 번만) → 설정 셀의 `KB`·`KB_TAG` 를 바꾸고 **런타임을 재시작**해 3절을 다시 실행 → 이 절의 셀을 실행한다. 결과가 없는 변형은 표에서 자동으로 빠진다.

In [ ]:
# 변형 B, C 검색 파일 생성 (GPU, 몇 분). 이미 있으면 건너뛴다. PEIR · ROCO 텍스트 피처는 Drive 캐시를 쓴다.
#  B: 이미지 쿼리 x KB 텍스트만 (train QA 는 이미지별로 합친 문장, 이미지↔이미지 유사도를 쓰지 않는다)
#  C: PEIR + ROCO 만 (train QA 없음 → 같은 이미지 · 같은 질문 유형 QA 가 들어올 수 없다)
import subprocess
def rv_build(kb, retrieval, suffix, extra=()):
    tag = '_'.join(kb.split(','))
    outs = [f'{R}/rag_{d}_{tag}{suffix}.json' for d in ('vqarad', 'slake')]
    if all(os.path.exists(o) for o in outs):
        print('있음, 건너뜀:', [os.path.basename(o) for o in outs]); return
    cmd = ['python', '-u', '/content/Expert-CFG/tools/build_rag_testjson.py',
           '--peir_dir', R, '--vqarad_common', '/content/drive/MyDrive/vqarad_common/testset.json',
           '--vqarad_train', '/content/drive/MyDrive/vqarad_common/trainset.json',
           '--vqarad_img', '/content/drive/MyDrive/vqa_rad/VQA_RAD Image Folder', '--out_dir', R,
           '--kb', kb, '--retrieval', retrieval, '--out_suffix', suffix, *extra]
    print(' '.join(cmd)); subprocess.run(cmd, check=True)
try:
    import open_clip  # noqa: F401
    _have_oc = True
except ImportError:
    _have_oc = False
if not _have_oc:
    print('open_clip_torch 가 없어 검색 파일 생성을 건너뜁니다. 이 세션은 transformers 5.x 호환 패치 환경이라 설치를 권하지 않습니다.')
    print('새 Colab 세션에서 README "2. RAG 검색 파일 만들기"의 방식으로 아래 두 변형을 만드세요 (이미 있으면 필요 없음).')
    print('  B: --retrieval cross --out_suffix _v2cross   /   C: --kb peir,roco --out_suffix _v2')
else:
    rv_build('peir,roco,trainqa', 'cross', '_v2cross', ('--trainqa', 'paper'))
    rv_build('peir,roco', 'cross', '_v2')


In [ ]:
# 4-2 로드 함수와 변형 정의 (GPU 불필요) — 이름 앞의 rv_ / RV_ 는 위쪽 셀의 변수와 겹치지 않게 하려는 접두어
import glob, pandas as pd
from collections import Counter
from IPython.display import display
RV_VARIANTS = {'A union+QA':         ('peir,roco,trainqa', '_v2'),
               'B 이미지→텍스트+QA':  ('peir,roco,trainqa', '_v2cross'),
               'C PEIR+ROCO':        ('peir,roco',         '_v2')}
RV_PAPER = {  # 논문 Table 1, Phi3.5V-Med (Open, Closed, Overall)
    'vqarad': {'baseline': (69.27, 90.44, 82.04), 'rag': (70.39, 88.24, 81.15), 'rag5': (70.95, 89.34, 82.04), 'expert_rag': (72.06, 90.44, 83.37), 'expert_cfg': (74.86, 91.18, 84.70)},
    'slake':  {'baseline': (82.95, 88.70, 85.20), 'rag': (82.79, 89.90, 85.58), 'rag5': (82.64, 89.42, 85.30), 'expert_rag': (83.56, 89.90, 86.05), 'expert_cfg': (85.27, 91.59, 87.75)}}[DATASET]
RV_ROWS = {'RAG-100':                   ('w/ RAG (top-1, 전 문항)',              'rag',        '규칙 준수'),
           'RAG-5':                     ('w/ 5% RAG',                           'rag5',       '규칙 준수'),
           'ExpertCFG-5-v2':            ('w/ 5% Expert-CFG (γ=1.3)',            'expert_cfg', '규칙 준수'),
           'ExpertCFG-5-v2-g15':        ('w/ 5% Expert-CFG (γ=1.5)',            'expert_cfg', '규칙 준수'),
           'ExpertRAG-5-inject-v2':     ('w/ 5% Expert-RAG (논문 방식: 전문가 주석=정답)', 'expert_rag', '규칙 완화'),
           'ExpertCFG-5-inject-v2':     ('w/ 5% Expert-CFG (논문 방식, γ=1.3)',  'expert_cfg', '규칙 완화'),
           'ExpertCFG-5-inject-v2-g15': ('w/ 5% Expert-CFG (논문 방식, γ=1.5)',  'expert_cfg', '규칙 완화')}

def rv_paths(kb, tag):
    suf = ('' if kb == 'peir' else '_' + kb.replace(',', '_')) + tag
    return f'{R}/rag_{DATASET}{suf}.json', f'{E}/ragcfg_{DATASET}{suf}'

def rv_load_baseline(rows):
    """저장된 베이스라인 예측을 rows 와 짝짓는다. 어긋나면 조용히 넘어가지 않고 멈춘다."""
    base_ = json.load(open(DS[DATASET]['baseline'], encoding='utf-8'))
    nk = lambda s_: str(s_).strip().lower()
    assert len(base_) == len(rows), (len(base_), len(rows))
    by_ = {b_['idx']: b_ for b_ in base_} if all('idx' in b_ for b_ in base_) else {i_: b_ for i_, b_ in enumerate(base_)}
    assert sorted(by_) == list(range(len(rows)))
    for i_, r_ in enumerate(rows):
        assert nk(by_[i_]['question']) == nk(r_['question']) and nk(by_[i_]['answer']) == nk(r_['answer']), f'베이스라인과 검색 파일의 {i_}번 문항이 다릅니다'
    return {i_: by_[i_]['prediction'] for i_ in by_}

def rv_load(kb, tag):
    rag_, out_ = rv_paths(kb, tag)
    if not (os.path.exists(rag_) and os.path.isdir(out_)):
        return None
    rows_ = json.load(open(rag_, encoding='utf-8'))
    runs_ = {}
    for f_ in sorted(glob.glob(f'{out_}/*.json')):
        nm_ = os.path.basename(f_)[:-5]
        if nm_ != 'summary':
            runs_[nm_] = {int(k_): v_ for k_, v_ in json.load(open(f_, encoding='utf-8')).items()}
    sel_ = sorted(runs_['RAG-5']) if 'RAG-5' in runs_ else None        # 게이팅으로 선택된 문항 = 5% 행이 저장한 문항
    return dict(rows=rows_, base=rv_load_baseline(rows_), runs=runs_, sel=sel_)

def rv_full(Vv, name):
    """행 이름의 전체 예측. 전 문항 행이 덜 끝났거나 게이팅 행이 선택 문항을 다 못 채웠으면 None (섞어 쓰지 않는다)."""
    p_ = Vv['runs'][name]
    if name == 'RAG-100':
        return p_ if len(p_) == len(Vv['rows']) else None
    if Vv['sel'] is None or set(p_) != set(Vv['sel']):
        return None
    return {**Vv['base'], **p_}

rv_recs = lambda rows_, preds_: [{'question': r_['question'], 'answer': r_['answer'], 'answer_type': r_['answer_type'], 'prediction': preds_[i_]} for i_, r_ in enumerate(rows_)]
rv_score = lambda rows_, preds_: np.array([vs.score_phi(x_) for x_ in rv_recs(rows_, preds_)])

def rv_stats(rows_):
    cap_ = lambda r_: r_['top_k_captions'][0].lower()
    op_ = [r_ for r_ in rows_ if r_['answer_type'] == 'OPEN']
    return {'top-1 출처': dict(Counter(r_.get('top_k_sources', ['?'])[0] for r_ in rows_)),
            '같은 이미지 train QA(top-1)': int(sum(bool(r_.get('top1_same_image')) for r_ in rows_)),
            'open 정답이 top-1 캡션에 그대로 있음': round(sum(str(r_['answer']).strip().lower() in cap_(r_) for r_ in op_) / max(1, len(op_)), 3),
            '질문이 top-1 캡션에 그대로 있음': round(sum(str(r_['question']).strip().rstrip('?').strip().lower() in cap_(r_) for r_ in rows_) / len(rows_), 3)}

RV = {n_: rv_load(kb_, tag_) for n_, (kb_, tag_) in RV_VARIANTS.items()}
for n_, v_ in RV.items():
    if v_ is None: print(f'[{n_}] 없음 — 검색 파일이나 결과 폴더가 아직 없음')
    else: print(f"[{n_}] 문항 {len(v_['rows'])} | 저장된 행: {sorted(v_['runs'])} | 5% 선택 {len(v_['sel']) if v_['sel'] else '미정'}문항")

In [ ]:
# 4-2 (1) RAG 하락 진단 — w/ RAG(전 문항) 대 베이스라인, 공식 채점. 원인은 단정하지 않고 분해 결과만 보여 준다.
def rv_diagnose(Vv, name='RAG-100', n_examples=8):
    rows_, base_ = Vv['rows'], Vv['base']
    new_ = rv_full(Vv, name)
    if new_ is None:
        print(f'{name} 이(가) 아직 전 문항이 끝나지 않았습니다 — 진단 건너뜀'); return None
    sb_, sn_ = rv_score(rows_, base_), rv_score(rows_, new_)
    cap_ = lambda r_: r_['top_k_captions'][0].lower()
    df_ = pd.DataFrame({
        'type': [r_['answer_type'] for r_ in rows_], 'base': sb_, 'new': sn_, 'delta': sn_ - sb_,
        'source': [r_.get('top_k_sources', ['?'])[0] for r_ in rows_],
        'same_img': [bool(r_.get('top1_same_image')) for r_ in rows_],
        'ans_in_cap': [str(r_['answer']).strip().lower() in cap_(r_) if r_['answer_type'] == 'OPEN' else np.nan for r_ in rows_],
        'q_in_cap': [str(r_['question']).strip().rstrip('?').strip().lower() in cap_(r_) for r_ in rows_]})
    agg_ = dict(n=('delta', 'size'), 베이스라인=('base', lambda x_: 100 * x_.mean()), RAG=('new', lambda x_: 100 * x_.mean()),
                Δ=('delta', lambda x_: 100 * x_.mean()), 나빠진=('delta', lambda x_: int((x_ < 0).sum())), 좋아진=('delta', lambda x_: int((x_ > 0).sum())))
    print(f'■ {name} 대 베이스라인 (공식 채점)')
    g_ = df_.groupby('type').agg(**agg_)
    g_.loc['전체'] = [len(df_), 100 * df_.base.mean(), 100 * df_.new.mean(), 100 * df_.delta.mean(), int((df_.delta < 0).sum()), int((df_.delta > 0).sum())]
    display(g_.round(2))
    for col_, title_ in (('same_img', '같은 이미지의 train QA 가 top-1 인가'), ('source', 'top-1 출처'),
                         ('ans_in_cap', 'open: 정답이 top-1 캡션에 있는가'), ('q_in_cap', '질문이 top-1 캡션에 그대로 있는가')):
        sub_ = df_ if col_ != 'ans_in_cap' else df_[df_.type == 'OPEN']
        print(f'\n■ {title_}'); display(sub_.groupby(col_).agg(**agg_).round(2))
    first_ = lambda p_: (str(p_).strip().lower().split() or [''])[0].strip('.,')
    cl_ = [i_ for i_, r_ in enumerate(rows_) if r_['answer_type'] == 'CLOSED']
    cat_ = lambda s_: s_ if s_ in ('yes', 'no') else '그 외'
    print('\n■ closed: 정답 분포와 예측 분포 (yes / no / 그 외)')
    display(pd.DataFrame({'정답': pd.Series([cat_(str(rows_[i_]['answer']).strip().lower()) for i_ in cl_]).value_counts(),
                          '베이스라인 예측': pd.Series([cat_(first_(base_[i_])) for i_ in cl_]).value_counts(),
                          'RAG 예측': pd.Series([cat_(first_(new_[i_])) for i_ in cl_]).value_counts()}).fillna(0).astype(int))
    print(f'\n■ 가장 크게 나빠진 {n_examples}문항')
    for i_ in df_.sort_values('delta').head(n_examples).index:
        r_ = rows_[i_]
        print(f"#{i_} [{r_['answer_type']}] 출처={df_.source[i_]} 같은이미지={df_.same_img[i_]} Δ={100 * df_.delta[i_]:+.0f}\n  Q: {r_['question']}\n"
              f"  정답: {r_['answer']} | 베이스라인: {base_[i_]!r} | RAG: {new_[i_]!r}\n  캡션: {r_['top_k_captions'][0][:260]}")
    return df_

RV_DIAG = {}
for n_, v_ in RV.items():
    if v_ is None: continue
    print('\n' + '=' * 30, n_, '=' * 30)
    print(rv_stats(v_['rows']))
    if 'RAG-100' in v_['runs']:
        RV_DIAG[n_] = rv_diagnose(v_)

In [ ]:
# 4-2 (2) 검색 방식 비교 — 변형 A / B / C 같은 표 (공식 채점, Δ 는 문항 쌍 부트스트랩 95% CI)
rv_rng = np.random.default_rng(0)
def rv_ci(new_, ref_, n_=10000):
    d_ = new_ - ref_
    return np.percentile(d_[rv_rng.integers(0, len(d_), (n_, len(d_)))].mean(1) * 100, [2.5, 97.5])

rv_main, rv_relaxed, rv_stat_rows, rv_base0 = [], [], [], None
for vname_, v_ in RV.items():
    if v_ is None: continue
    sb_ = rv_score(v_['rows'], v_['base'])
    if rv_base0 is None:
        m0_ = vs.summarize(rv_recs(v_['rows'], v_['base']), vs.score_phi); rv_base0 = (m0_['open'], m0_['closed'], m0_['overall'])
    rv_stat_rows.append({'변형': vname_, **{k_: (str(x_) if isinstance(x_, dict) else x_) for k_, x_ in rv_stats(v_['rows']).items()}})
    for key_, (label_, pk_, rule_) in RV_ROWS.items():
        if key_ not in v_['runs']: continue
        p_ = rv_full(v_, key_)
        if p_ is None:
            print(f'[{vname_}] {key_}: 아직 끝나지 않음 ({len(v_["runs"][key_])}문항 저장) — 표에서 제외'); continue
        m_ = vs.summarize(rv_recs(v_['rows'], p_), vs.score_phi)
        arr_ = rv_score(v_['rows'], p_); lo_, hi_ = rv_ci(arr_, sb_)
        (rv_relaxed if rule_ == '규칙 완화' else rv_main).append({
            '변형': vname_, '행': label_, 'open': m_['open'], 'closed': m_['closed'], 'overall': m_['overall'],
            'Δ 대 베이스라인': f"{(arr_ - sb_).mean() * 100:+.2f} [{lo_:+.2f}, {hi_:+.2f}]",
            '논문 Δ': f"{RV_PAPER[pk_][2] - RV_PAPER['baseline'][2]:+.2f}", '논문 overall': RV_PAPER[pk_][2]})

def rv_show(rows_, title_):
    if not rows_: print(title_, '— 없음'); return
    print(title_); st_ = pd.DataFrame(rows_).style.format({c_: '{:.2f}' for c_ in ('open', 'closed', 'overall', '논문 overall')})
    try: st_ = st_.hide(axis='index')
    except Exception: pass
    display(st_)

if rv_base0:
    print(f'[{DATASET}] 베이스라인(공식): open {rv_base0[0]:.2f} / closed {rv_base0[1]:.2f} / overall {rv_base0[2]:.2f} | 논문 {RV_PAPER["baseline"]}')
rv_show(rv_stat_rows, '\n■ 검색 구성 (변형별)')
rv_show(rv_relaxed, '\n■ 논문 방식 행 (기본 비교) — 전문가 주석 = 정답. 규칙 완화(oracle): test 정답이 입력에 들어가므로 규칙 준수 결과로 보고하지 않는다')
rv_show(rv_main, '\n■ 규칙 준수 행 (정답 미사용; 논문 방식 행과 입력 정보가 다르다)')
fn_ = f'{E}/followup_{DATASET}.json'
json.dump({'dataset': DATASET, 'gate': GATE_ENTROPY, 'baseline_official': list(rv_base0) if rv_base0 else None, 'paper': RV_PAPER,
           'rule_compliant': rv_main, 'rule_relaxed': rv_relaxed, 'retrieval': rv_stat_rows}, open(fn_, 'w', encoding='utf-8'), ensure_ascii=False, indent=1, default=str)
print('\n저장:', fn_)

## 5. 종합 비교 (두 데이터셋)

In [ ]:
from google.colab import drive; drive.mount('/content/drive')

# 종합 비교 — VQA-RAD 와 SLAKE 를 논문과 함께 (GPU 불필요; 두 데이터셋을 각각 한 번씩 실행한 뒤 사용)
import json, os
E = '/content/drive/MyDrive/expert_cfg'
KB_SUFFIX = '_peir_roco_trainqa_v2'        # '' = PEIR only / '_peir_roco_trainqa' 등 = 다른 KB 의 결과를 볼 때
S = {}
for d in ('vqarad', 'slake'):
    p = f'{E}/ragcfg_{d}{KB_SUFFIX}/summary.json'
    if os.path.exists(p):
        S[d] = json.load(open(p, encoding='utf-8'))
        print(f"{d}: {p}\n     KB = {S[d].get('kb', '(기록 없음 — 이 파일은 KB 기록 기능 추가 전에 만들어짐. 폴더 이름으로 판단)')} | 검색 파일 = {S[d].get('rag_file', '(기록 없음)')}")
    else: print(f'{d}: 아직 실행 결과 없음 ({p})')
NAMES = {'baseline': '베이스라인', 'RAG-100': 'w/ RAG', 'RAG-5': '5% RAG', 'ExpertCFG-5-v2': '5% Expert-CFG (논문 식 7~10)',
         'ExpertRAG-5-inject-v2': '5% Expert-RAG (정답 주입)', 'ExpertCFG-5-inject-v2': '5% Expert-CFG (정답 주입)',
         'ExpertCFG-5-v2-g15': '5% Expert-CFG (γ=1.5)', 'ExpertCFG-5-inject-v2-g15': '5% Expert-CFG (정답 주입, γ=1.5)'}
if len({s.get('kb', '?') for s in S.values()}) > 1:
    print('\n[경고] 두 데이터셋의 KB 가 다릅니다 — 같은 KB 끼리 비교해야 합니다 (KB_SUFFIX 를 확인하세요)')
import pandas as pd
from IPython.display import display
def _show(rows, title, fmt):
    print(title)
    st = pd.DataFrame(rows).style.format(fmt, na_rep='')
    try: st = st.hide(axis='index')
    except Exception: pass
    display(st.set_properties(**{'text-align': 'right'}).set_properties(subset=['행', '구분'], **{'text-align': 'left'}))

if len(S) == 2:
    keys = [k for k in S['vqarad']['rows'] if k in S['slake']['rows'] and k in NAMES]
    abs_rows, delta_rows = [], []
    for k in keys:
        grp = '규칙 완화' if '-inject' in k else '규칙 준수'
        a_row, d_row = {'행': NAMES[k], '구분': grp}, {'행': NAMES[k], '구분': grp}
        for d, tag in (('vqarad', 'VQA-RAD'), ('slake', 'SLAKE')):
            rr = S[d]['rows']; v = rr[k]['원본']['overall']; pv = rr[k]['paper'][2]
            vb, pb = rr['baseline']['원본']['overall'], rr['baseline']['paper'][2]
            a_row[f'{tag} 우리'], a_row[f'{tag} 논문'] = v, pv
            if k != 'baseline':
                d_row[f'{tag} 우리 Δ'], d_row[f'{tag} 논문 Δ'] = v - vb, pv - pb
        abs_rows.append(a_row)
        if k != 'baseline': delta_rows.append(d_row)
    _show(abs_rows, '\n■ overall (우리 = 공식 채점(원본), 논문 = Phi3.5V-Med)', {c_: '{:.2f}' for c_ in abs_rows[0] if c_ not in ('행', '구분')})
    _show(delta_rows, '\n■ 베이스라인 대비 변화 (overall)', {c_: '{:+.2f}' for c_ in delta_rows[0] if c_ not in ('행', '구분')})
    print('\n※ 규칙 완화 = open 캡션에 정답을 덧붙인 상한선. KB 가 다르면 비교할 수 없으므로 위 KB 출력을 먼저 확인할 것.')
else:
    print('두 데이터셋의 summary.json 이 모두 있어야 표를 만듭니다 (위 출력 확인)')


In [ ]:
# AIMS_shared 1차 업로드 — 어댑터(체크포인트)와 베이스라인 예측. Colab 한 셀로 끝난다 (레포 클론 불필요).
# 규칙: 릴리스 폴더는 한 번 올리면 덮어쓰지 않는다(같은 내용이면 건너뛰고, 다르면 멈춘다). 학습 상태(optimizer.pt 등)는 올리지 않는다.
import os, json, shutil, hashlib, datetime
from google.colab import drive
drive.mount('/content/drive')

E = '/content/drive/MyDrive/expert_cfg'
S = '/content/drive/MyDrive/AIMS_shared'
INCLUDE = ['vqarad', 'slake']
SPEC = {
    'vqarad': dict(name='phi35-vqarad-e10-s42-v1', pred_name='phi35-vqarad-baseline-ckpt257-v1',
                   ckpt=f'{E}/logs_phi35_vqarad_continue_e10_env_common/checkpoint-257',
                   pred=f'{E}/vqarad_continue_e10_env_common_ckpt257_test.json',
                   note='train만 학습, val로 고른 체크포인트(step 257). Phi-3.5V-Med LoRA 위에 이어서 학습'),
    'slake':  dict(name='phi35-slake-e5-s42-v1', pred_name='phi35-slake-baseline-ckpt380-v1',
                   ckpt=f'{E}/logs_phi35_slake_continue_e5_env_common/checkpoint-380',
                   pred=f'{E}/slake_continue_e5_env_common_ckpt380_test.json',
                   note='train만 학습, val로 고른 체크포인트(step 380). Phi-3.5V-Med LoRA 위에 이어서 학습'),
}
ADAPTER_FILES = ('adapter_model.safetensors', 'adapter_config.json')    # run_config.json 은 있으면 함께

def sha256(p, bs=1 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for c in iter(lambda: f.read(bs), b''):
            h.update(c)
    return h.hexdigest()

def hash_folder(d):
    return {os.path.relpath(os.path.join(r, f), d).replace(os.sep, '/'): sha256(os.path.join(r, f))
            for r, _, fs in os.walk(d) for f in sorted(fs)}

def release(dest, files):
    """files: {릴리스 안의 이름: 원본 경로}. 이미 있으면 내용이 같을 때만 건너뛰고, 다르면 멈춘다 (덮어쓰지 않음)."""
    if os.path.isdir(dest):
        want = {k: sha256(v) for k, v in files.items()}
        have = hash_folder(dest)
        assert want == have, f'{dest} 이(가) 이미 있고 내용이 다릅니다. 덮어쓰지 않습니다 — 이름 끝을 -v2 로 바꿔 새로 올리세요.'
        print('  이미 같은 내용으로 있음, 건너뜀:', dest); return have
    os.makedirs(dest)
    for k, v in files.items():
        shutil.copy2(v, os.path.join(dest, k))
    return hash_folder(dest)

manifest = {}
for key in INCLUDE:
    sp = SPEC[key]; ck = sp['ckpt']
    assert os.path.isdir(ck), f'없음: {ck}'
    # 1) 어댑터
    files = {}
    for f in ADAPTER_FILES:
        assert os.path.exists(f'{ck}/{f}'), f'없음: {ck}/{f}'
        files[f] = f'{ck}/{f}'
    for cand in (f'{ck}/run_config.json', f'{os.path.dirname(ck)}/run_config.json'):
        if os.path.exists(cand): files['run_config.json'] = cand; break
    else:
        print(f'  [{key}] run_config.json 없음 — 어댑터 두 파일만 올립니다')
    print(f"[{key}] 올리지 않는 파일(학습 상태 등): {sorted(set(os.listdir(ck)) - set(files))}")
    rel = f"checkpoints/phi35/{sp['name']}"
    manifest[sp['name']] = {'where': f'AIMS_shared/{rel}', 'files': release(f'{S}/{rel}', files), 'protocol': 'compliant',
                            'note': sp['note'], 'recorded': datetime.date.today().isoformat()}
    # 2) 베이스라인 예측 — 엔트로피 필드는 뺀다 (저장값은 <|end|> 포함 정의라 공식 불확실성이 아니다. 불확실성은 별도 파일로 올린다)
    recs = json.load(open(sp['pred'], encoding='utf-8'))
    drop = sorted({k for r in recs for k in r if 'entropy' in k.lower()})
    clean = [{k: v for k, v in r.items() if k not in drop} for r in recs]
    tmp = f'/content/{sp["pred_name"]}_test_predictions.json'
    json.dump(clean, open(tmp, 'w', encoding='utf-8'), ensure_ascii=False)
    rel = f"baselines/{sp['pred_name']}"
    manifest[sp['pred_name']] = {'where': f'AIMS_shared/{rel}', 'files': release(f'{S}/{rel}', {'test_predictions.json': tmp}),
                                 'protocol': 'compliant', 'note': f'베이스라인 test 예측 {len(clean)}문항, 제거한 필드: {drop}. 어댑터: {sp["name"]}',
                                 'recorded': datetime.date.today().isoformat()}
    print(f'[{key}] 예측 {len(clean)}문항, 제거한 필드 {drop}, 남은 필드 {sorted(clean[0])}')

readme = f'{S}/README.txt'
if not os.path.exists(readme):
    open(readme, 'w', encoding='utf-8').write(
        'AIMS_shared — 공유 폴더. 릴리스 폴더(-v1)는 한 번 올리면 덮어쓰지 않는다. 바뀌면 -v2 로 새로 올린다.\n'
        'checkpoints/<phi3|phi35>/<이름>-v1/  어댑터(adapter_model.safetensors, adapter_config.json, run_config.json)\n'
        'baselines/<이름>-v1/                 베이스라인 test 예측 (불확실성 필드 없음)\n'
        'uncertainty/<이름>-v1/               불확실성(엔트로피) 파일 — 정의와 계산에 쓴 예측·어댑터를 파일 안에 적는다\n'
        'rag_assets/<이름>-v1/                검색 파일\n'
        'results/<날짜>_<이름>/               실행 결과\n'
        '자세한 규칙: AIMS_basement 레포의 manifests/README.md\n')
print('\n=== 아래 JSON 을 그대로 복사해 주세요 (레포의 manifests/checkpoints.json 에 기록합니다) ===')
print(json.dumps(manifest, indent=2, ensure_ascii=False, sort_keys=True))


## 해석 시 주의

1. **검색은 자체 구현이고 논문 KB와 같지 않다.** KB는 PEIR 33,572 + ROCO 65,422(공개본, 논문 70,306) + train QA(이미지별로 합쳐 723항목, `no` 제외, PathVQA 없음) = 99,717이다. 논문의 train QA 문장화(GPT-4)와 쿼리 종류(Image/Text/Sum/Union 중 메인에 쓴 것)는 원문에 없거나 우리가 쓸 수 없다. **union 검색에서는 top-1이 거의 항상 train QA이고 PEIR·ROCO는 사실상 쓰이지 않는다**(VQA-RAD 100%, SLAKE 99.9%). 그래서 w/ RAG 행의 논문 대비 격차는 검색 차이를 포함하고, 4-2 절에서 변형 A/B/C를 같은 표로 비교한다.
2. **5% 게이팅의 표본이 작다.** VQA-RAD 는 23문항, SLAKE 는 53문항만 바뀐다. overall 의 변화는 표의 부트스트랩 CI 와 함께 읽고, CI 가 0 을 포함하면 "차이 없음"으로 본다.
3. **정답 주입 행은 규칙 완화다.** 정답 정보가 들어가므로 규칙 준수 결과로 보고하지 않는다. 형식은 위 "정답 주입 형식" 참고. 옛 KB(VQA-RAD, `peir,roco,trainqa` v1)에서는 선택 open 20문항의 정답이 캡션에 한 번도 없어서 정답 하이라이트만으로는 마스크가 생기지 않았다(그래서 "하이라이트=정답" 행은 뺐다). v2(union + train QA)에서는 SLAKE open 정답이 top-1 캡션에 있는 비율이 60.6%라 정답이 이미 캡션에 있는 경우가 많고, 이때는 정답을 덧붙이지 않아 마스크가 캡션 속 기존 위치에 걸린다. 논문의 큰 개선폭은 전문가가 참조 텍스트에 정답을 넣는 방식이어야 설명되므로, 정답 주입 행이 그 상한선이다.
4. **공식 채점은 원본(Phi 3.5V 방식)이다.** S1+S2 는 Phi 의 VQA-RAD test 오답에서 만든 규칙이라 교수님 규칙 기준으로 규칙 완화에 해당하므로 참고 열로만 둔다.
5. Expert-RAG 의 조건부 가지 attention 배율(`cond_boost`)은 논문 코드에서 확인된 정의가 아니라 이 노트북의 구현이다.
6. **정답 주입 행과 규칙 준수 행은 closed 문항의 하이라이트 규칙이 다르다**(위 표). 두 행의 차이에는 정답 정보 외에 이 차이가 섞여 있다.
7. **게이팅 정의(`eq4` / `stored`)는 이 노트북의 결과 폴더 이름에 `_gate_stored` 로만 구분된다.** 이 구분이 생기기 전에 만든 결과 폴더는 `summary.json` 에 정의가 기록되어 있지 않다. 이후 실행은 `summary.json` 에 `gate` 가 기록된다.
8. **환경:** 이 노트북의 결과는 transformers 5.x 호환 패치(12~16번 셀) 환경에서 나왔다. 팀 공통 환경(transformers 4.44.2)에서는 스모크 점검(베이스라인 50문항 재현, 훅 점검)만 통과했고 RAG · CFG 결과를 재현하지는 않았다.
9. **기본 비교 행(논문 방식, 정답 주입)은 정보가 다른 두 조건을 비교하는 것이다.** 정답을 쓰는 행과 쓰지 않는 행의 차이에는 정답 정보가 섞여 있어, 정답 주입 행의 점수를 방법의 성능으로 읽지 않고 "전문가 주석이 주어졌을 때의 상한선"으로 읽는다.
